In [1]:
import os
from pathlib import Path
# 创建项目目录结构
# 获取当前脚本所在目录的绝对路径
current_dir = Path.cwd()  # 或者 Path(".").absolute()
project_root = current_dir / "ship_detection"
dataset_dir = project_root / "datasets" / "seaships"
models_dir = project_root / "models"
results_dir = project_root / "results"

print(f"项目根目录: {project_root}")

项目根目录: D:\ShipTarget\ship_detection


In [2]:
from ultralytics import YOLO
import yaml

# 加载数据集配置
with open(dataset_dir / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

print("数据集配置:")
print(f"类别数量: {data_config['nc']}")
print(f"类别名称: {data_config['names']}")
print(f"训练图像路径: {dataset_dir / data_config['train']}")
print(f"验证图像路径: {dataset_dir / data_config['val']}")

# 验证目录是否存在
train_img_dir = dataset_dir / data_config['train']
assert train_img_dir.exists(), f"训练图像目录不存在: {train_img_dir}"

数据集配置:
类别数量: 1
类别名称: ['ship']
训练图像路径: D:\ShipTarget\ship_detection\datasets\seaships\images\train
验证图像路径: D:\ShipTarget\ship_detection\datasets\seaships\images\val


In [4]:
# 加载预训练模型
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 设置 HF 镜像
model = YOLO('yolov10s.pt')  # 将自动下载预训练权重
print("模型加载成功")

模型加载成功


In [5]:
# 开始训练
results = model.train(
    data=str(dataset_dir / "data.yaml"),  # 数据集配置
    epochs=100,                            # 训练轮数（根据开题报告）
    imgsz=512,                              # 输入图像尺寸
    batch=8,                                # 批次大小（根据显存调整）
    workers=2,                                # 数据加载线程数
    device=0,                                 # cpu
    project=str(results_dir / "yolov10"),    # 结果保存目录
    name="baseline512",                          # 实验名称
    exist_ok=True,                             # 允许覆盖
    patience=20,                                # 早停耐心值
    save=True,                                   # 保存模型
    save_period=10,                               # 每10轮保存一次
    plots=True,                                    # 生成训练图表
    cache=False,                                     # 缓存数据加速关闭，减轻压力
)

print("YOLOv11训练完成")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\seaships\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, m

In [6]:
# 在测试集上评估模型
best_model_path = results_dir / "yolov10" / "baseline512" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='test',  # 使用测试集
    batch=16,
    imgsz=640,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLOv10s summary (fused): 106 layers, 7,218,387 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 481.8180.8 MB/s, size: 172.4 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\seaships\labels\test.cache... 1018 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1018/1018  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 5.6it/s 11.4s0.2s
                   all       1018       2865      0.636       0.51      0.599      0.395
Speed: 1.2ms preprocess, 7.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val29
YOLOv11评估结果:
mAP50: 0.5993
mAP50-95: 0.3947
召回率: 0.5103
精确率: 0.6365


In [7]:
# 加载预训练模型
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 设置 HF 镜像
model = YOLO('yolov8s.pt')  # 将自动下载预训练权重
print("模型加载成功")

模型加载成功


In [8]:
# 开始训练
results = model.train(
    data=str(dataset_dir / "data.yaml"),  # 数据集配置
    epochs=100,                            # 训练轮数（根据开题报告）
    imgsz=512,                              # 输入图像尺寸
    batch=8,                                # 批次大小（根据显存调整）
    workers=2,                                # 数据加载线程数
    device=0,                                 # cpu
    project=str(results_dir / "yolov8"),    # 结果保存目录
    name="baseline512",                          # 实验名称
    exist_ok=True,                             # 允许覆盖
    patience=20,                                # 早停耐心值
    save=True,                                   # 保存模型
    save_period=10,                               # 每10轮保存一次
    plots=True,                                    # 生成训练图表
    cache=False,                                     # 缓存数据加速关闭，减轻压力
)

print("YOLOv11训练完成")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\seaships\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mo

In [9]:
# 在测试集上评估模型
best_model_path = results_dir / "yolov8" / "baseline512" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='test',  # 使用测试集
    batch=16,
    imgsz=640,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 420.196.6 MB/s, size: 168.5 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\seaships\labels\test.cache... 1018 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1018/1018  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 5.8it/s 11.1s0.2s
                   all       1018       2865      0.638      0.567      0.631      0.404
Speed: 0.8ms preprocess, 7.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val30
YOLOv11评估结果:
mAP50: 0.6315
mAP50-95: 0.4035
召回率: 0.5668
精确率: 0.6381


In [10]:
import os
from pathlib import Path
# 创建项目目录结构
# 获取当前脚本所在目录的绝对路径
current_dir = Path.cwd()  # 或者 Path(".").absolute()
project_root = current_dir / "ship_detection"
dataset_dir = project_root / "datasets" / "SSDD"
models_dir = project_root / "models"
results_dir = project_root / "results"

print(f"项目根目录: {project_root}")

项目根目录: D:\ShipTarget\ship_detection


In [11]:
from ultralytics import YOLO
import yaml

# 加载数据集配置
with open(dataset_dir / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

print("数据集配置:")
print(f"类别数量: {data_config['nc']}")
print(f"类别名称: {data_config['names']}")
print(f"训练图像路径: {dataset_dir / data_config['train']}")
print(f"验证图像路径: {dataset_dir / data_config['val']}")

# 验证目录是否存在
train_img_dir = dataset_dir / data_config['train']
assert train_img_dir.exists(), f"训练图像目录不存在: {train_img_dir}"

数据集配置:
类别数量: 1
类别名称: ['ship']
训练图像路径: D:\ShipTarget\ship_detection\datasets\SSDD\images\train
验证图像路径: D:\ShipTarget\ship_detection\datasets\SSDD\images\val


In [12]:
# 加载预训练模型
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 设置 HF 镜像
model = YOLO('yolov10s.pt')  # 将自动下载预训练权重
print("模型加载成功")

模型加载成功


In [13]:
# 开始训练
results = model.train(
    data=str(dataset_dir / "data.yaml"),  # 数据集配置
    epochs=100,                            # 训练轮数（根据开题报告）
    imgsz=320,                              # 输入图像尺寸
    batch=8,                                # 批次大小（根据显存调整）
    workers=2,                                # 数据加载线程数
    device=0,                                 # cpu
    project=str(results_dir / "yolov10"),    # 结果保存目录
    name="baseline_SSDD",                          # 实验名称
    exist_ok=True,                             # 允许覆盖
    patience=20,                                # 早停耐心值
    save=True,                                   # 保存模型
    save_period=10,                               # 每10轮保存一次
    plots=True,                                    # 生成训练图表
    cache=False,                                     # 缓存数据加速关闭，减轻压力
)

print("YOLOv11训练完成")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\SSDD\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosai

In [14]:
# 在测试集上评估模型
best_model_path = results_dir / "yolov10" / "baseline_SSDD" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='val',  # 使用测试集
    batch=2,
    imgsz=320,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLOv10s summary (fused): 106 layers, 7,218,387 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 548.6214.7 MB/s, size: 48.4 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\SSDD\labels\val.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 48.6it/s 2.4s0.1s
                   all        232        546      0.969      0.914       0.96      0.712
Speed: 0.3ms preprocess, 6.4ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val31
YOLOv11评估结果:
mAP50: 0.9600
mAP50-95: 0.7120
召回率: 0.9143
精确率: 0.9689


In [15]:
# 加载预训练模型
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 设置 HF 镜像
model = YOLO('yolov8s.pt')  # 将自动下载预训练权重
print("模型加载成功")

模型加载成功


In [16]:
# 开始训练
results = model.train(
    data=str(dataset_dir / "data.yaml"),  # 数据集配置
    epochs=100,                            # 训练轮数（根据开题报告）
    imgsz=320,                              # 输入图像尺寸
    batch=8,                                # 批次大小（根据显存调整）
    workers=2,                                # 数据加载线程数
    device=0,                                 # cpu
    project=str(results_dir / "yolov"),    # 结果保存目录
    name="baseline_SSDD",                          # 实验名称
    exist_ok=True,                             # 允许覆盖
    patience=20,                                # 早停耐心值
    save=True,                                   # 保存模型
    save_period=10,                               # 每10轮保存一次
    plots=True,                                    # 生成训练图表
    cache=False,                                     # 缓存数据加速关闭，减轻压力
)

print("YOLOv11训练完成")

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\ShipTarget\ship_detection\datasets\SSDD\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic

In [17]:
# 在测试集上评估模型
best_model_path = results_dir / "yolov" / "baseline_SSDD" / "weights" / "best.pt"
model = YOLO(best_model_path)

# 评估
metrics = model.val(
    data=str(dataset_dir / "data.yaml"),
    split='val',  # 使用测试集
    batch=2,
    imgsz=320,
    conf=0.25,     # 置信度阈值
    iou=0.5,       # IoU阈值
)

print("YOLOv11评估结果:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"召回率: {metrics.box.mr:.4f}")
print(f"精确率: {metrics.box.mp:.4f}")

Ultralytics 8.4.16  Python-3.11.14 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 481.2141.4 MB/s, size: 44.9 KB)
val: Scanning D:\ShipTarget\ship_detection\datasets\SSDD\labels\val.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 116/116 56.0it/s 2.1s0.0s
                   all        232        546      0.926      0.927       0.96      0.693
Speed: 0.5ms preprocess, 4.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to D:\ShipTarget\runs\detect\val32
YOLOv11评估结果:
mAP50: 0.9600
mAP50-95: 0.6930
召回率: 0.9267
精确率: 0.9262
